# G2 – Validação independente com os dados de 09/11/2026

Complemento do `processamento_dispersao_colab.ipynb` (sensor A = ESP-D68785). A calibração é a do relatório (Seção 3.1): $C = k\,\Delta\kappa$, com $k = 0{,}31$ mg/L por µS/cm na sonda A e $0{,}37$ na sonda E.

1. **Curva projetada a partir dos dados iniciais** de cada ensaio, sem usar as curvas de traçador: massa injetada (16 g), vazão do Venturi (Eq. 7), largura e nível d'água, distância de cada sensor, $U = Q/(BH)$ e $D_L$ de Fischer (Eq. 16), na solução analítica (Eq. 12), com o instante de injeção da Eq. (15).
2. **Curva medida** nos sensores A e E, convertida em concentração pela calibração de cada sonda.
3. **Propagação** da curva medida em A até E com $U$ e $D_L$ do routing A→C (Eq. 14).
4. **Degraus de recirculação**: as injeções 1 e 2 lançaram a mesma massa na mesma água, então os incrementos de concentração devem ser iguais.

Saídas: `figuras/F18_validacao_projecao_E.pdf` (e .png) e tabelas em `tabelas/`.

## 1. Arquivo de dados

In [ ]:
import os, subprocess, sys

LINK_SENSOR_DATA = "https://drive.google.com/file/d/1PtcX1IXsd1fZRJ0pMwMyCOU1-S4wnPna/view?usp=sharing"
ARQUIVO = "sensor_data.csv"

if not os.path.exists(ARQUIVO):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
    try:
        gdown.download(LINK_SENSOR_DATA, ARQUIVO, quiet=False, fuzzy=True)
    except Exception as erro:
        print("Não foi possível baixar do Drive:", erro)
if not os.path.exists(ARQUIVO):
    from google.colab import files
    nome = list(files.upload().keys())[0]
    os.rename(nome, ARQUIVO)
os.makedirs("figuras", exist_ok=True)
os.makedirs("tabelas", exist_ok=True)
print("ok:", ARQUIVO)

## 2. Código
Cada célula grava um módulo `.py`. `g2lib.py` tem as mesmas etapas do notebook principal (reamostragem, controle de qualidade, intercalibração, pulsos, momentos e routing).

**`estilo.py`** – Estilo das figuras.

In [ ]:
%%writefile estilo.py
"""Estilo das figuras do relatório (mesma paleta e tipografia das figuras anteriores)."""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

TINTA, TINTA2, GRADE, EIXO = "#191919", "#666666", "#e7e7e7", "#c2c2c2"
CORES = {"A": "#257de0", "B": "#f47230", "C": "#00b894", "D": "#f2a600", "E": "#ee7fac"}
MARC = {"A": "o", "B": "s", "C": "^", "D": "D", "E": "v"}
AZUL, LARANJA, VERDE, AMARELO, MAGENTA, VIOLETA, VERMELHO = (
    "#257de0", "#f47230", "#00b894", "#f2a600", "#ee7fac", "#6a4ce2", "#ec4647")

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9.5, "text.color": TINTA, "axes.labelcolor": TINTA2,
    "xtick.color": TINTA2, "ytick.color": TINTA2, "axes.edgecolor": EIXO, "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": GRADE, "grid.linewidth": 0.6, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False, "axes.titlesize": 9.5, "axes.titlelocation": "left",
    "savefig.dpi": 200, "lines.linewidth": 1.6, "axes.axisbelow": True,
})


def _fmt(v, p=None):
    s = f"{v:g}"
    return s.replace(".", ",").replace("-", "−")


VIRG = FuncFormatter(_fmt)


def virgula(ax, x=True, y=True):
    if x:
        ax.xaxis.set_major_formatter(VIRG)
    if y:
        ax.yaxis.set_major_formatter(VIRG)


def num(v, casas=1):
    """Número com vírgula decimal (para rótulos)."""
    return f"{v:.{casas}f}".replace(".", ",").replace("-", "−")


def salvar(fig, caminho_sem_ext):
    fig.savefig(caminho_sem_ext + ".pdf", bbox_inches="tight")
    fig.savefig(caminho_sem_ext + ".png", bbox_inches="tight", dpi=170)
    plt.close(fig)


**`g2lib.py`** – Processamento do ensaio de 09/11.

In [ ]:
%%writefile g2lib.py
"""Funções comuns do processamento do G2 (ensaio em canal de 09/11/2026).

Reproduz as etapas descritas nos Métodos do relatório (Seção 2.8.2): reamostragem em 1 s,
controle de qualidade, intercalibração, isolamento dos pulsos, momentos temporais e routing.
"""
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

if not hasattr(np, "trapezoid"):          # NumPy < 2.0
    np.trapezoid = np.trapz

# ---------------------------------------------------------------- arranjo de 09/11
DIA = "2026-11-09"
PLACAS = {"A": "ESP-D68785", "B": "ESP-848B36", "C": "ESP-4EE77C", "D": "ESP-104D45", "E": "ESP-C9C537"}
SENSORES = list(PLACAS)
X_CM = {"A": 91.15, "B": 91.15, "C": 146.0, "D": 147.0, "E": 233.0}     # distância ao lançamento
H_CM = {"A": 18.0, "B": 6.5, "C": 21.5, "D": 8.6, "E": 11.0}            # altura acima do fundo
ENSAIOS = pd.DataFrame({
    "ensaio": [1, 2, 3, 4, 5],
    "injecao": pd.to_datetime([f"{DIA} {h}" for h in ["16:08", "16:19", "16:29", "16:41", "16:52"]]),
    "theta": [0.0, 1.0, 2.0, 2.5, 2.3],
    "dh_cm": [24.5, 22.0, 22.0, 22.0, 25.3],
    "nivel_cm": [22.5, 21.0, np.nan, 12.0, 16.4],        # ensaio 3: 18,7 a 23,5 cm entre as seções
})
B_CM = 14.5                    # largura do canal
M_SAL_G = 16.0                 # massa de NaCl por injeção (125 mL a 128 kg/m3)
C_INJ_MG_L = 128000.0
FRAC_CL = 35.45 / 58.44
INICIO_REF = pd.Timestamp(f"{DIA} 16:03:47")
FIM_REF = pd.Timestamp(f"{DIA} 16:07:50")
FIM_C = pd.Timestamp(f"{DIA} 16:55:40")   # sonda C fora da água a partir daqui


def ler_sensor_data(caminho):
    d = pd.read_csv(caminho)
    d["t"] = pd.to_datetime(d["time"], dayfirst=True)
    return d


def serie_1s(d, placa, col, inicio, fim):
    g = d[d.board_id == placa].copy()
    g["seg"] = g.t.dt.floor("s")
    s = g.groupby("seg")[col].mean()
    return s.reindex(pd.date_range(inicio, fim, freq="1s"))


def preencher_lacunas(s, max_lacuna=5):
    """Interpola linearmente só as lacunas com até max_lacuna valores faltando."""
    nul = s.isna()
    grupos = (nul != nul.shift()).cumsum()
    tam = nul.groupby(grupos).transform("sum")
    interp = s.interpolate(method="time", limit_area="inside")
    return s.where(~(nul & (tam <= max_lacuna)), interp)


def preparar(d):
    """Séries de 1 s por sensor (condutividade, temperatura e tensão), com o controle de qualidade."""
    inicio, fim = d.t.min().floor("s"), d.t.max().floor("s")
    out, falhas = {}, {}
    for p, placa in PLACAS.items():
        k = serie_1s(d, placa, "conductivity_us_cm", inicio, fim)
        T = serie_1s(d, placa, "temperature_c", inicio, fim)
        V = serie_1s(d, placa, "voltage_v", inicio, fim)
        med = k.rolling(61, center=True, min_periods=10).median()
        ruim = k < 0.75 * med
        if p == "C":
            ruim |= (k.index >= FIM_C) & k.notna()
        falhas[p] = int(ruim.sum())
        k, T, V = k.mask(ruim), T.mask(ruim), V.mask(ruim)
        df = pd.DataFrame({"kappa": preencher_lacunas(k), "T": preencher_lacunas(T), "V": preencher_lacunas(V)})
        df["Vc"] = df.V / (1 + 0.02 * (df["T"] - 25))           # tensão corrigida para 25 °C
        out[p] = df
    return out, falhas


def referencia(series):
    lin = []
    for p, df in series.items():
        k = df.kappa[INICIO_REF:FIM_REF].dropna()
        lin.append(dict(pos=p, n=len(k), media=k.mean(), dp=k.std(), cv=100 * k.std() / k.mean(),
                        media_Vc=df.Vc[INICIO_REF:FIM_REF].mean()))
    r = pd.DataFrame(lin).set_index("pos")
    r["f"] = r.media.mean() / r.media
    r["fV"] = r.media_Vc.mean() / r.media_Vc
    return r


# ---------------------------------------------------------------- pulsos
def janela(series, p, inj, antes=60, depois=360, col="kappa"):
    s = series[p][col][inj - pd.Timedelta(seconds=antes): inj + pd.Timedelta(seconds=depois)]
    return s.interpolate(method="time", limit_area="inside")


def delimitar(dk_suave, inj, limiar=0.05, busca_pico=150, n_abaixo=5, max_fim=60):
    """Início, pico e fim do pulso (índices de tempo) sobre o excesso suavizado."""
    t_rel = (dk_suave.index - inj).total_seconds()
    zona = dk_suave[(t_rel >= 0) & (t_rel <= busca_pico)]
    tp = zona.idxmax()
    pico = zona.max()
    lim = limiar * pico
    antes = dk_suave[:tp]
    abaixo = antes[antes < lim]
    ti = abaixo.index[-1] if len(abaixo) else antes.index[0]
    depois = dk_suave[tp:]
    tf = None
    ab = (depois < lim).values
    for i in range(len(ab) - n_abaixo + 1):
        if ab[i:i + n_abaixo].all():
            tf = depois.index[i]
            break
    if tf is None or (tf - tp).total_seconds() > max_fim:
        trecho = depois[: tp + pd.Timedelta(seconds=max_fim)]
        tf = trecho.iloc[1:].idxmin()
    return ti, tp, tf, pico


def momentos(c, t):
    m0 = np.trapezoid(c, t)
    tm = np.trapezoid(t * c, t) / m0
    var = np.trapezoid((t - tm) ** 2 * c, t) / m0
    return m0, tm, var


def pulso(series, p, inj, base=None, mm=5, limiar=0.05, depois=360, col="kappa"):
    """Excesso, delimitação e momentos de um pulso. base: valor de referência (padrão: mediana de 60 s antes)."""
    s = janela(series, p, inj, depois=depois, col=col)
    if base is None:
        base = s[inj - pd.Timedelta(seconds=60): inj - pd.Timedelta(seconds=1)].median()
    dk = s - base
    dks = dk.rolling(mm, center=True, min_periods=1).mean()
    ti, tp, tf, pico = delimitar(dks, inj, limiar=limiar)
    c = dks[ti:tf].clip(lower=0)
    t = (c.index - inj).total_seconds().values
    m0, tm, var = momentos(c.values, t)
    return dict(pos=p, base=base, dk=dk, dks=dks, ti=ti, tp=tp, tf=tf, pico=pico,
                M0=m0, tm=tm, var=var, sig=np.sqrt(var), dur=(tf - ti).total_seconds(),
                t_ini=(ti - inj).total_seconds(), t_pico=(tp - inj).total_seconds(), t_fim=(tf - inj).total_seconds())


def curva_pulso(r, inj, t_grade):
    """Curva do pulso (suavizada, zero fora do trecho do pulso) numa grade de tempo relativa à injeção."""
    c = r["dks"][r["ti"]:r["tf"]].clip(lower=0)
    t = (c.index - inj).total_seconds().values
    return np.interp(t_grade, t, c.values, left=0.0, right=0.0)


# ---------------------------------------------------------------- routing (Fischer, 1968)
def propagar(t, c1, U, D, dx):
    """Propaga a curva c1(t) (grade uniforme) por dx com a solução da ADE (Eq. routing)."""
    dt = t[1] - t[0]
    tp = dx / U
    out = np.zeros_like(c1, dtype=float)
    for j, tj in enumerate(t):
        arg = tp - tj + t
        out[j] = np.sum(c1 * U / np.sqrt(4 * np.pi * D * tp) * np.exp(-(U * arg) ** 2 / (4 * D * tp))) * dt
    return out


def r2_pulso(obs, prev, frac=0.01):
    m = (obs > frac * obs.max()) | (prev > frac * prev.max())
    ss_res = np.sum((obs[m] - prev[m]) ** 2)
    ss_tot = np.sum((obs[m] - obs[m].mean()) ** 2)
    return 1 - ss_res / ss_tot


def ajustar_routing(t, c1, c2, dx, U0, D0):
    a1, a2 = np.trapezoid(c1, t), np.trapezoid(c2, t)
    n1, n2 = c1 / a1, c2 / a2

    def res(p):
        return propagar(t, n1, p[0], p[1], dx) - n2

    sol = least_squares(res, [U0, max(D0, 1.0)], bounds=([0.2, 0.05], [50, 500]))
    U, D = sol.x
    prev = propagar(t, n1, U, D, dx)
    return U, D, r2_pulso(n2, prev), prev, n1, n2


**`validacao_independente.py`** – Projeção, comparação, propagação, degraus de recirculação e figura.

In [ ]:
%%writefile validacao_independente.py
"""Validação independente com os dados do ensaio de 09/11: curva projetada × curva medida.

Uso: python3 validacao_independente.py sensor_data.csv pasta_figuras pasta_tabelas

1. Projeção a partir dos dados iniciais de cada ensaio, sem usar as curvas de traçador: massa injetada
   M = 16 g, vazão Q do Venturi (Eq. 7), largura B = 14,5 cm, nível d'água H anotado, distância x de cada
   sensor ao lançamento, U = Q/(B H) e D_L pela expressão de Fischer (Eq. 16). A curva projetada é a
   solução analítica (Eq. 12), com o instante de injeção da Eq. (15).
2. Curva medida: excesso de condutividade convertido em concentração pela calibração de cada sonda
   (Seção 3.1): k = 0,309 mg/L por µS/cm na sonda A e 0,37 na sonda E.
3. Propagação da curva medida em A até E com U e D_L do routing A->C (Eq. 14).
4. Degraus de recirculação: as injeções 1 e 2 lançaram a mesma massa na mesma água, então os
   incrementos de concentração (Eq. 5) devem ser iguais.
"""
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).parent))
import estilo as E
from estilo import plt
import g2lib as L

K_CAL = {"A": 0.309, "E": 0.37}          # mg/L por µS/cm (calibração de 05/10, Seção 3.1)
H_ENSAIO = {1: 22.5, 2: 21.0, 3: (18.7 + 23.5) / 2, 4: 12.0, 5: 16.4}   # nível d'água anotado (cm)
NU = 1.03e-6                             # m²/s, água a 19 °C
D1, D2, RHO = 0.024, 0.0136, 13.6        # Venturi (m) e densidade relativa do mercúrio

# degraus de recirculação: janelas com o mesmo atraso após as injeções 1 e 2 (+3 a +5,5 min)
T = lambda h: pd.Timestamp(f"{L.DIA} {h}")
JANELAS = {"P1": ("16:11:00", "16:13:30"), "P2": ("16:22:00", "16:24:30"),
           "B1": ("16:07:00", "16:07:59"), "B2": ("16:18:00", "16:18:59")}
T_P1, T_B2, T_P2 = 4.25, 10.5, 15.25     # min após a injeção 1 (centro das janelas)
SENS_TESTE = ["B", "C", "D", "E"]        # A: manuseado entre 16:24 e 16:26:40 (avaliado à parte)


def vazao(dh_cm, cd=1.0):
    """Eq. (7): vazão do Venturi (m³/s)."""
    a2 = np.pi * D2 ** 2 / 4
    return cd * a2 * np.sqrt(2 * 9.81 * dh_cm / 100 * (RHO - 1) / (1 - (D2 / D1) ** 4))


def fischer(U, B, H):
    """Eq. (16) com u* de parede lisa (Blasius). U, B, H em m -> D_L (m²/s) e u* (m/s)."""
    Rh = B * H / (B + 2 * H)
    f = 0.316 * (4 * U * Rh / NU) ** -0.25
    us = U * np.sqrt(f / 8)
    return 0.011 * U ** 2 * B ** 2 / (H * us), us


def solucao_analitica(x_cm, t_s, M_g, A_cm2, U, D):
    """Eq. (12): concentração média na seção (mg/L); x em cm, t em s após a injeção, U em cm/s, D em cm²/s."""
    t = np.maximum(np.asarray(t_s, float), 1e-6)
    c = M_g * 1000 / (A_cm2 * np.sqrt(4 * np.pi * D * t)) * np.exp(-(x_cm - U * t) ** 2 / (4 * D * t))
    return c * 1000


def nse_pulso(obs, sim, frac=0.01):
    """NSE no trecho do pulso (onde a curva medida ou a projetada passa de 1 % do pico)."""
    m = (obs > frac * obs.max()) | (sim > frac * sim.max())
    o, s = obs[m], sim[m]
    return 1 - np.sum((o - s) ** 2) / np.sum((o - o.mean()) ** 2)


def analisar(S, e):
    inj = e.injecao
    rk = {p: L.pulso(S, p, inj) for p in L.SENSORES}
    UAC = (L.X_CM["C"] - L.X_CM["A"]) / (rk["C"]["tm"] - rk["A"]["tm"])
    t0 = round(rk["A"]["tm"] - L.X_CM["A"] / UAC)                 # Eq. (15), s após o minuto anotado
    H = H_ENSAIO[e.ensaio]
    Q = vazao(e.dh_cm)
    Ua = 100 * Q / (L.B_CM / 100 * H / 100)
    Da, us = fischer(Ua / 100, L.B_CM / 100, H / 100)
    Da *= 1e4
    A_cm2 = L.B_CM * H
    tg = np.arange(-30, 330, 1.0)                                 # s após a injeção estimada
    curvas, lin = {}, []
    for p in ["A", "E"]:
        r = rk[p]
        c = (K_CAL[p] * r["dks"][r["ti"]:r["tf"]]).clip(lower=0)
        t = (c.index - inj).total_seconds().values - t0
        cm = np.interp(tg, t, c.values, left=0.0, right=0.0)
        cp = solucao_analitica(L.X_CM[p], tg, L.M_SAL_G, A_cm2, Ua, Da)
        m0, tm, var = L.momentos(cm, tg)
        m0p, tmp_, varp = L.momentos(cp, tg)
        curvas[p] = dict(t=tg, med=cm, proj=cp)
        lin.append(dict(ensaio=e.ensaio, pos=p, k=K_CAL[p], pico_med=cm.max(), pico_proj=cp.max(),
                        razao_pico=cm.max() / cp.max(), tm_med=tm, tm_proj=tmp_, dtm=tm - tmp_,
                        sig_med=np.sqrt(var), sig_proj=np.sqrt(varp), area_med=m0, area_proj=m0p,
                        recuperacao=Q * 1000 * m0 / (L.M_SAL_G * 1000), nse_forma=nse_pulso(cm / m0, cp / m0p)))
    # propagação da curva medida em A até E (Eq. 14), com U e D_L do routing A->C
    tr = np.arange(-60, 361, 1.0)
    ca, cc = L.curva_pulso(rk["A"], inj, tr), L.curva_pulso(rk["C"], inj, tr)
    dx = L.X_CM["C"] - L.X_CM["A"]
    dtm = rk["C"]["tm"] - rk["A"]["tm"]
    U0, D0 = dx / dtm, (dx / dtm) ** 2 / 2 * (rk["C"]["var"] - rk["A"]["var"]) / dtm
    Ur, Dr, r2r, *_ = L.ajustar_routing(tr, ca, cc, dx, U0, D0)
    cA = K_CAL["A"] * ca
    cEp = np.interp(tg, tr - t0, L.propagar(tr, cA, Ur, Dr, L.X_CM["E"] - L.X_CM["A"]), left=0.0, right=0.0)
    cmE = curvas["E"]["med"]
    m0e, tme, _ = L.momentos(cmE, tg)
    m0q, tmq, _ = L.momentos(cEp, tg)
    curvas["E"]["prop_A"] = cEp
    prop = dict(ensaio=e.ensaio, U_routing=Ur, D_routing=Dr, tm_prop=tmq, dtm_prop=tme - tmq, pico_prop=cEp.max(),
                razao_pico_prop=cmE.max() / cEp.max(), razao_area_E_A=m0e / np.trapezoid(cA, tr),
                nse_forma_prop=nse_pulso(cmE / m0e, cEp / m0q))
    info = dict(ensaio=int(e.ensaio), theta=float(e.theta), t0_s=t0, Q_L_s=1000 * Q, H_cm=H, U_proj=Ua, D_proj=Da,
                u_estrela_cm_s=100 * us, tempo_viagem_E_s=L.X_CM["E"] / Ua)
    return lin, prop, info, curvas


def degraus_recirculacao(S, Rref, k):
    """Incrementos de concentração das injeções 1 e 2 (mesma massa), com a calibração linear C = k Δκ."""
    out = []
    for p in L.SENSORES:
        s = S[p].kappa * Rref.f[p] / Rref.f["A"]                    # escala da sonda A (Eq. 8)
        v = {kk: s[T(a):T(b)].median() for kk, (a, b) in JANELAS.items()}
        Ex = {kk: k * (v[kk] - v["B1"]) for kk in ["P1", "B2", "P2"]}
        lam = np.log(Ex["P1"] / Ex["B2"]) / (T_B2 - T_P1)              # queda entre as injeções (1/min)
        e1, e2 = Ex["P1"], Ex["P2"] - Ex["B2"] * np.exp(-lam * (T_P2 - T_B2))
        out.append(dict(pos=p, e1=e1, e2=e2, razao=e2 / e1, razao_sem_decaimento=(Ex["P2"] - Ex["B2"]) / e1,
                        queda_pct_min=100 * lam))
    return pd.DataFrame(out).set_index("pos")


if __name__ == "__main__":
    arq, sfig, stab = sys.argv[1], Path(sys.argv[2]), Path(sys.argv[3])
    sfig.mkdir(parents=True, exist_ok=True)
    stab.mkdir(parents=True, exist_ok=True)
    S, _ = L.preparar(L.ler_sensor_data(arq))
    Rref = L.referencia(S)
    LIN, PROP, INFO, CUR = [], [], [], {}
    for _, e in L.ENSAIOS.iterrows():
        lin, prop, info, cur = analisar(S, e)
        LIN += lin
        PROP.append(prop)
        INFO.append(info)
        CUR[e.ensaio] = cur
    Tb, Pb, Ib = pd.DataFrame(LIN), pd.DataFrame(PROP), pd.DataFrame(INFO)
    Tb.round(3).to_csv(stab / "projecao_x_medida_A_E.csv", index=False)
    Pb.round(3).to_csv(stab / "propagacao_A_para_E.csv", index=False)
    Ib.round(3).to_csv(stab / "projecao_parametros.csv", index=False)
    inc = degraus_recirculacao(S, Rref, K_CAL["A"])
    inc.round(3).to_csv(stab / "degraus_recirculacao.csv")
    sel = inc.loc[SENS_TESTE]
    rec = dict(razao_media_BE=sel.razao.mean(), razao_dp_BE=sel.razao.std(), razao_min=sel.razao.min(),
               razao_max=sel.razao.max(), razao_sem_queda_BE=sel.razao_sem_decaimento.mean(), razao_A=inc.razao["A"],
               e1_min=sel.e1.min(), e1_max=sel.e1.max(), e2_min=sel.e2.min(), e2_max=sel.e2.max(),
               e1_media=sel.e1.mean(), volume_L=L.M_SAL_G * 1000 / sel.e1.mean(),
               queda_min=sel.queda_pct_min.min(), queda_max=sel.queda_pct_min.max())
    json.dump(dict(parametros=INFO, recirculacao=rec), open(stab / "validacao_resumo.json", "w"), indent=1, default=float)
    pd.set_option("display.width", 250)
    print(Ib.round(2).to_string(index=False))
    print(Tb.round(2).to_string(index=False))
    print(Pb.round(2).to_string(index=False))
    print(inc.round(3).to_string())
    print({kk: round(float(v), 3) for kk, v in rec.items()})

    # ---------------------------------------------------------------- figura
    fig, axs = plt.subplots(2, 5, figsize=(11.0, 5.0))
    for j, (_, e) in enumerate(L.ENSAIOS.iterrows()):
        c = CUR[e.ensaio]["E"]
        t = c["t"]
        xmax = 100 if e.ensaio == 4 else 150
        ax = axs[0, j]
        ax.plot(t, c["proj"], color=E.TINTA, lw=1.3, ls="--", label="Projetada a partir dos dados iniciais (16 g)")
        ax.plot(t, c["med"], color=E.CORES["E"], lw=2.0, label="Medida em E (calibração da sonda E)")
        ax.set_title(f"Exp. {e.ensaio} ({E.num(e.theta, 1)}°)", loc="center", fontsize=9)
        ax.set_xlim(0, xmax)
        ax.set_ylim(0, None)
        E.virgula(ax)
        ax = axs[1, j]
        ax.plot(t, c["med"] / np.trapezoid(c["med"], t), color=E.CORES["E"], lw=2.0)
        ax.plot(t, c["proj"] / np.trapezoid(c["proj"], t), color=E.TINTA, lw=1.3, ls="--")
        ax.plot(t, c["prop_A"] / np.trapezoid(c["prop_A"], t), color=E.CORES["A"], lw=1.4, ls=":",
                label="Propagada a partir da curva medida em A")
        r = Tb[(Tb.ensaio == e.ensaio) & (Tb.pos == "E")].iloc[0]
        q = Pb[Pb.ensaio == e.ensaio].iloc[0]
        ax.text(0.97, 0.96, f"NSE proj. {E.num(r.nse_forma, 2)}\nNSE prop. {E.num(q.nse_forma_prop, 2)}",
                transform=ax.transAxes, ha="right", va="top", fontsize=6.8, color=E.TINTA2)
        ax.set_xlim(0, xmax)
        ax.set_ylim(0, None)
        ax.set_xlabel("t (s)")
        E.virgula(ax)
    axs[0, 0].set_ylabel("NaCl em excesso (mg/L)")
    axs[1, 0].set_ylabel("Normalizada (1/s)")
    h1, l1 = axs[0, 0].get_legend_handles_labels()
    h2, l2 = axs[1, 0].get_legend_handles_labels()
    fig.legend(h1 + h2, l1 + l2, loc="lower center", ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.01))
    fig.tight_layout(rect=(0, 0.06, 1, 1), h_pad=1.0, w_pad=0.6)
    E.salvar(fig, str(sfig / "F18_validacao_projecao_E"))


## 3. Execução

In [ ]:
!python validacao_independente.py sensor_data.csv figuras tabelas

## 4. Resultados

In [ ]:
import pandas as pd, json
from IPython.display import Image, display
t = pd.read_csv("tabelas/projecao_x_medida_A_E.csv")
p = pd.read_csv("tabelas/propagacao_A_para_E.csv")
display(pd.read_csv("tabelas/projecao_parametros.csv").round(2))
display(t[["ensaio", "pos", "pico_med", "pico_proj", "recuperacao", "dtm", "sig_med", "sig_proj", "nse_forma"]].round(2))
display(p[["ensaio", "U_routing", "D_routing", "dtm_prop", "razao_area_E_A", "nse_forma_prop"]].round(2))
print(json.load(open("tabelas/validacao_resumo.json"))["recirculacao"])
display(Image("figuras/F18_validacao_projecao_E.png", width=900))

## 5. Download

In [ ]:
import zipfile, glob
with zipfile.ZipFile("validacao_independente.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("figuras/*") + glob.glob("tabelas/*") + glob.glob("*.py"):
        z.write(f)
try:
    from google.colab import files
    files.download("validacao_independente.zip")
except ImportError:
    print("Arquivo gerado: validacao_independente.zip")